In [117]:
import logging
import pandas as pd
import numpy as np
import os

logging.basicConfig(level=logging.INFO)

class FileValidationError(Exception):
    pass

class FileMissingValueError(FileValidationError):
    pass

def load_results(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contains no value or data")
    else:
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        logging.info(f"Loaded {filepath} — shape: {df.shape}")
        return df

try:
    al = load_results("data/raw/Allocated Limit for Honble MPs.csv")
    wc = load_results("data/raw/Works Completed.csv")
    ws = load_results("data/raw/Works Sanctioned.csv")

except FileValidationError as e:
    logging.error(f"Loading Failed: {e}")
    raise

INFO:root:Loaded data/raw/Allocated Limit for Honble MPs.csv — shape: (544, 5)
INFO:root:Loaded data/raw/Works Completed.csv — shape: (15001, 11)
INFO:root:Loaded data/raw/Works Sanctioned.csv — shape: (6001, 12)


In [118]:
import re

def clean_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    
    dfCols = list(df.columns)
    columns_cleaned = []

    for col in dfCols:
        cleaned_names = re.sub("[\.\'\()\₹]", '',str(col)).lower().strip()
        cleaned_names = cleaned_names.replace(" ", "_")
        columns_cleaned.append(cleaned_names)

    df.columns = (columns_cleaned)
    return df


clean_dataframe_columns(al).columns
clean_dataframe_columns(wc).columns
clean_dataframe_columns(ws).columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida',
       'honble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
      dtype='object')

In [114]:
al.columns, ws.columns, wc.columns

(Index(['sr_no', 'state', 'honble_members_of_parliaments', 'constituency',
        'allocated_amount'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida',
        'honble_members_of_parliament', 'constituency', 'work_description',
        'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida', 'work_description',
        'honble_members_of_parliament', 'constituency', 'image',
        'completion_date', 'amount_disbursed'],
       dtype='object'))

In [166]:
def diagnose_dataframe(df: pd.DataFrame, name: str):
    logging.info(f"\n Dataframe -> {name}\n")
    logging.info(f" Shape: \n{df.shape}\n")
    logging.info(f" Dataframe Datatypes \n{df.dtypes}\n")
    logging.info(f" Dataframe null columns \n{df.isna().sum()[df.isna().sum()>0]}\n")
    logging.info(f" Datarframe duplicates \n{df.duplicated().sum()}")

    
diagnose_dataframe(al, "Allocated Limit for Honble MPs")
diagnose_dataframe(wc, "Works Completed")
diagnose_dataframe(ws, "Works Sanctioned")

INFO:root:
 Dataframe -> Allocated Limit for Honble MPs

INFO:root: Shape: 
(544, 5)

INFO:root: Dataframe Datatypes 
sr_no                            object
state                            object
honble_members_of_parliaments    object
constituency                     object
allocated_amount                 object
dtype: object

INFO:root: Dataframe null columns 
allocated_amount    1
dtype: int64

INFO:root: Datarframe duplicates 
0
INFO:root:
 Dataframe -> Works Completed

INFO:root: Shape: 
(15001, 11)

INFO:root: Dataframe Datatypes 
sr_no                           object
work_category                   object
work                            object
state                           object
ida                             object
work_description                object
honble_members_of_parliament    object
constituency                    object
image                           object
completion_date                 object
amount_disbursed                object
dtype: object

INFO:root:

In [167]:
print(al['sr_no'].head())
print(wc['sr_no'].head())
print(ws['sr_no'].head())

print(wc['image'].sample(10))
print(wc['work_description'].sample(10))
print(ws['work_description'].sample(10))

0    1
1    2
2    3
3    4
4    5
Name: sr_no, dtype: object
0    1
1    2
2    3
3    4
4    5
Name: sr_no, dtype: object
0    1
1    2
2    3
3    4
4    5
Name: sr_no, dtype: object
3845     Images
2954     Images
3502     Images
5325        NaN
13417    Images
10716    Images
10850       NaN
3463        NaN
1108        NaN
8206     Images
Name: image, dtype: object
9376     Improvement of water harvesting at Tharvelangs...
12689             GOVT HSS KARDA ME LAB SETUP INSTALLATION
9356     Installation work of a semi high mast light 6 ...
2702     Parori Block khurja me Devi mandir se Ramesh D...
272         Construction of Community hall in Reddy Colony
4636     HIGH MAST LIGHT WITH SIX LED RECOMMENDED IN MY...
350      C/o RLR at Ekisang Village in Lower Dibang Val...
12441    Gram Sabha Keshav Raypur me Ravindra Patel ke ...
10977    BAISA PRAKHAND ANTARGART RAUTA PANCHAYAT KE KA...
9152                    Construction public park at Jevari
Name: work_description, dtype: object

In [ ]:
# ============================================
# STEP 5 — OBSERVATIONS (Task 1)
# ============================================

# --- Allocated Limit for Honble MPs ---
# - All columns are dtype object, including allocated_amount (should be numeric)
# - allocated_amount has 1 null value
# - sr_no is a redundant sequential counter stored as text — will drop in Task 2
# - No "N/A" string issue found
# - No date columns in this table
# - 0 duplicate rows

# --- Works Sanctioned ---
# - All columns are dtype object, including sanction_amount (should be numeric)
# - work_description has 38 nulls
# - recommended_date and sanction_date are stored as object, need datetime conversion
# - sr_no is a redundant sequential counter — will drop in Task 2
# - Checked work_description for "N/A" strings — none found, looks like genuine free text
# - Inconsistent casing in work_description (ALL CAPS / Title Case / lowercase mixed) — 
#   not a join key, so leaving as-is, just noting it
# - 0 duplicate rows

# --- Works Completed ---
# - All columns are dtype object, including amount_disbursed (should be numeric)
# - work_description has 55 nulls, image has 5054 nulls, amount_disbursed has 6 nulls
# - completion_date is stored as object, needs datetime conversion
# - sr_no is a redundant sequential counter — will drop in Task 2
# - image column checked — no "N/A" strings, but values are either real NaN or the 
#   placeholder text "Images" (non-informative) — will drop this column in Task 2
# - 0 duplicate rows